# Tahap 05 — BERTopic Modeling

## Judul Project

**Analisis Komputasional Topik Pidato Presiden Prabowo pada Forum Nasional dan Internasional Menggunakan Manual Coding dan BERTopic**

## Tujuan Notebook

Notebook ini digunakan untuk menjalankan **BERTopic Modeling** terhadap data chunk pidato yang sudah dipersiapkan pada Tahap 03.

Tahap ini mencakup:

1. Membaca dataset chunk untuk BERTopic.
2. Membaca hasil manual coding sebagai referensi filtering dokumen non-substantif.
3. Melakukan validasi struktur data.
4. Menyiapkan konfigurasi BERTopic untuk dataset kecil.
5. Membuat embedding multilingual.
6. Melatih model BERTopic.
7. Mengekstrak topik, keyword, dan distribusi dokumen.
8. Menyimpan output hasil topic modeling.
9. Membuat visualisasi dasar BERTopic.
10. Menyiapkan output untuk Tahap 06, yaitu perbandingan Manual Coding dan BERTopic.

## Input Utama

```text
data/processed/speech_chunks_for_bertopic.csv
data/processed/speech_chunks_master.csv
data/processed/manual_coding_final.csv
```

## Output Utama

```text
data/processed/bertopic_document_topics.csv
data/processed/bertopic_topic_info.csv
data/processed/bertopic_topic_keywords.csv
data/processed/bertopic_document_topics_with_manual_reference.csv
reports/tables/bertopic_modeling_summary.csv
reports/tables/bertopic_topic_distribution.csv
reports/tables/stage05_output_manifest.json
reports/figures/bertopic_topic_barchart.html
reports/figures/bertopic_topics_overview.html
```

## Catatan Metodologis

BERTopic pada tahap ini digunakan sebagai pendekatan **unsupervised topic modeling**.  
Hasil manual coding tidak digunakan untuk melatih model, tetapi digunakan sebagai referensi untuk mengecualikan chunk yang sudah ditandai `EXCLUDED`, misalnya chunk pembuka, salam, atau daftar sapaan yang tidak substantif.

## 1. Import Library Dasar

Cell ini memuat library dasar yang dibutuhkan untuk membaca data, validasi, dan penyimpanan output.

In [1]:
# ============================================================
# Import Library Dasar
# ============================================================

from pathlib import Path
from datetime import datetime
import importlib
import json
import re
import subprocess
import sys
import warnings

import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")

print("Library dasar berhasil di-import.")
print(f"Python version: {sys.version}")
print(f"Pandas version: {pd.__version__}")
print(f"Numpy version: {np.__version__}")

Library dasar berhasil di-import.
Python version: 3.10.20 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:42:35) [MSC v.1942 64 bit (AMD64)]
Pandas version: 2.3.3
Numpy version: 2.2.6


## 2. Setup Path Project

Notebook akan mendeteksi root project berdasarkan keberadaan file:

```text
data/processed/speech_chunks_for_bertopic.csv
```

In [2]:
# ============================================================
# Setup Path Project
# ============================================================

def find_project_root(start_path=None):
    """
    Mendeteksi root folder project berdasarkan keberadaan file input Tahap 05.
    """
    if start_path is None:
        start_path = Path.cwd().resolve()
    else:
        start_path = Path(start_path).resolve()

    candidate_paths = [start_path] + list(start_path.parents)

    for candidate in candidate_paths:
        expected_input = candidate / "data" / "processed" / "speech_chunks_for_bertopic.csv"
        if expected_input.exists():
            return candidate

    return start_path


PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
INTERIM_DIR = DATA_DIR / "interim"
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORT_TABLE_DIR = REPORTS_DIR / "tables"
REPORT_FIGURE_DIR = REPORTS_DIR / "figures"
MODELS_DIR = PROJECT_ROOT / "models"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
REPORT_TABLE_DIR.mkdir(parents=True, exist_ok=True)
REPORT_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

INPUT_CHUNKS_BERTOPIC = PROCESSED_DIR / "speech_chunks_for_bertopic.csv"
INPUT_CHUNKS_MASTER = PROCESSED_DIR / "speech_chunks_master.csv"
INPUT_MANUAL_CODING_FINAL = PROCESSED_DIR / "manual_coding_final.csv"

print("Project path berhasil disiapkan.")
print(f"Current working directory   : {Path.cwd().resolve()}")
print(f"PROJECT_ROOT                : {PROJECT_ROOT}")
print(f"INPUT_CHUNKS_BERTOPIC       : {INPUT_CHUNKS_BERTOPIC}")
print(f"INPUT_CHUNKS_MASTER         : {INPUT_CHUNKS_MASTER}")
print(f"INPUT_MANUAL_CODING_FINAL   : {INPUT_MANUAL_CODING_FINAL}")
print(f"REPORT_TABLE_DIR            : {REPORT_TABLE_DIR}")
print(f"REPORT_FIGURE_DIR           : {REPORT_FIGURE_DIR}")

Project path berhasil disiapkan.
Current working directory   : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\notebooks
PROJECT_ROOT                : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech
INPUT_CHUNKS_BERTOPIC       : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\processed\speech_chunks_for_bertopic.csv
INPUT_CHUNKS_MASTER         : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\processed\speech_chunks_master.csv
INPUT_MANUAL_CODING_FINAL   : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\processed\manual_coding_final.csv
REPORT_TABLE_DIR            : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\reports\tables
REPORT_FIGURE_DIR           : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\reports\figures


## 3. Preflight Check File Input

Cell ini memastikan file utama tersedia sebelum proses modeling dijalankan.

In [3]:
# ============================================================
# Preflight Check Input
# ============================================================

if not INPUT_CHUNKS_BERTOPIC.exists():
    raise FileNotFoundError(
        f"File input utama tidak ditemukan: {INPUT_CHUNKS_BERTOPIC}\n"
        "Pastikan Tahap 03 sudah dijalankan dan file speech_chunks_for_bertopic.csv "
        "tersimpan di data/processed/."
    )

if not INPUT_CHUNKS_MASTER.exists():
    print(
        "Peringatan: speech_chunks_master.csv tidak ditemukan. "
        "Notebook tetap dapat berjalan menggunakan speech_chunks_for_bertopic.csv."
    )
else:
    print("File speech_chunks_master.csv ditemukan.")

if not INPUT_MANUAL_CODING_FINAL.exists():
    print(
        "Peringatan: manual_coding_final.csv tidak ditemukan. "
        "Filtering chunk EXCLUDED tidak akan diterapkan."
    )
else:
    print("File manual_coding_final.csv ditemukan.")

print("Preflight check selesai.")
print(f"Ukuran speech_chunks_for_bertopic.csv: {INPUT_CHUNKS_BERTOPIC.stat().st_size:,} bytes")

File speech_chunks_master.csv ditemukan.
File manual_coding_final.csv ditemukan.
Preflight check selesai.
Ukuran speech_chunks_for_bertopic.csv: 123,269 bytes


## 4. Membaca Dataset

Dataset utama untuk modeling adalah `speech_chunks_for_bertopic.csv`.

Jika tersedia, `manual_coding_final.csv` dibaca untuk mengecualikan chunk dengan `coding_status = EXCLUDED`.

In [4]:
# ============================================================
# Load Dataset
# ============================================================

chunks_bertopic_df = pd.read_csv(INPUT_CHUNKS_BERTOPIC)

chunks_master_df = None
if INPUT_CHUNKS_MASTER.exists():
    chunks_master_df = pd.read_csv(INPUT_CHUNKS_MASTER)

manual_coding_final_df = None
if INPUT_MANUAL_CODING_FINAL.exists():
    manual_coding_final_df = pd.read_csv(INPUT_MANUAL_CODING_FINAL)

print("Dataset utama berhasil dibaca.")
print(f"chunks_bertopic_df shape: {chunks_bertopic_df.shape}")

if chunks_master_df is not None:
    print(f"chunks_master_df shape: {chunks_master_df.shape}")

if manual_coding_final_df is not None:
    print(f"manual_coding_final_df shape: {manual_coding_final_df.shape}")

display(chunks_bertopic_df.head())

print("Daftar kolom chunks_bertopic_df:")
for col in chunks_bertopic_df.columns:
    print(f"- {col}")

Dataset utama berhasil dibaca.
chunks_bertopic_df shape: (74, 11)
chunks_master_df shape: (74, 25)
manual_coding_final_df shape: (74, 34)


,doc_id,speech_id,file_name,speech_title_from_filename,forum_scope_inferred,event_date,language_estimate,chunk_order,text,chunk_word_count,chunk_quality_flags
0,SPCH_001_BRICS_LEADERS_CHK_001,SPCH_001_BRICS_LEADERS,NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt,Brics Leaders,international,2025-09-08,en,1,Distinguished Leaders of BRICS. It is indeed a...,227,OK
1,SPCH_001_BRICS_LEADERS_CHK_002,SPCH_001_BRICS_LEADERS,NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt,Brics Leaders,international,2025-09-08,en,2,"We consider now, this is the time that BRICS m...",42,LOW_WORD_COUNT_LT_80
2,SPCH_002_PANEN_RAYA_CHK_001,SPCH_002_PANEN_RAYA,NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt,Panen Raya,national,2026-01-07,id,1,Bismillahirrahmanirrahim. Assalamu'alaikum war...,268,OK
3,SPCH_002_PANEN_RAYA_CHK_002,SPCH_002_PANEN_RAYA,NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt,Panen Raya,national,2026-01-07,id,2,"Yang saya hormati, para Dirut BUMN yang berken...",224,OK
4,SPCH_002_PANEN_RAYA_CHK_003,SPCH_002_PANEN_RAYA,NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt,Panen Raya,national,2026-01-07,id,3,"Dirut PT Berdikari Saudara Maryadi, Dirut Sang...",242,OK


Daftar kolom chunks_bertopic_df:
- doc_id
- speech_id
- file_name
- speech_title_from_filename
- forum_scope_inferred
- event_date
- language_estimate
- chunk_order
- text
- chunk_word_count
- chunk_quality_flags


## 5. Validasi Kolom Wajib

Kolom wajib untuk BERTopic:

- `doc_id`
- `speech_id`
- `file_name`
- `text`
- `chunk_word_count`

Kolom `doc_id` harus unik karena menjadi identitas dokumen.

In [5]:
# ============================================================
# Helper Validasi Kolom
# ============================================================

def require_columns(df, required_columns, df_name="DataFrame"):
    """
    Memastikan DataFrame memiliki kolom yang dibutuhkan.
    """
    missing_columns = [col for col in required_columns if col not in df.columns]

    if missing_columns:
        raise ValueError(
            f"{df_name} tidak memiliki kolom wajib: {missing_columns}. "
            f"Kolom tersedia: {list(df.columns)}"
        )


def safe_value_counts(df, column_name):
    """
    Value counts aman jika kolom tersedia.
    """
    if column_name not in df.columns:
        return pd.DataFrame(columns=[column_name, "count"])

    return (
        df[column_name]
        .fillna("MISSING")
        .replace("", "MISSING")
        .value_counts()
        .reset_index()
        .rename(columns={"index": column_name, column_name: "count"})
    )


REQUIRED_BERTOPIC_COLUMNS = [
    "doc_id",
    "speech_id",
    "file_name",
    "text",
    "chunk_word_count"
]

require_columns(chunks_bertopic_df, REQUIRED_BERTOPIC_COLUMNS, "chunks_bertopic_df")

if chunks_bertopic_df["doc_id"].isna().any():
    raise ValueError("Terdapat doc_id kosong pada chunks_bertopic_df.")

if chunks_bertopic_df["doc_id"].duplicated().any():
    duplicated_doc_ids = chunks_bertopic_df.loc[
        chunks_bertopic_df["doc_id"].duplicated(),
        "doc_id"
    ].tolist()
    raise ValueError(f"Terdapat doc_id duplikat: {duplicated_doc_ids}")

if chunks_bertopic_df["text"].isna().any():
    raise ValueError("Terdapat text kosong pada chunks_bertopic_df.")

if (chunks_bertopic_df["text"].astype(str).str.strip() == "").any():
    raise ValueError("Terdapat text kosong setelah strip pada chunks_bertopic_df.")

if (chunks_bertopic_df["chunk_word_count"] <= 0).any():
    raise ValueError("Terdapat chunk_word_count tidak valid.")

if manual_coding_final_df is not None:
    REQUIRED_MANUAL_COLUMNS = [
        "chunk_id",
        "manual_open_code_1",
        "manual_axial_category",
        "manual_selective_theme",
        "coding_status"
    ]
    require_columns(manual_coding_final_df, REQUIRED_MANUAL_COLUMNS, "manual_coding_final_df")

print("Validasi kolom wajib berhasil.")
print(f"Jumlah dokumen chunk: {len(chunks_bertopic_df)}")
print(f"Jumlah pidato unik: {chunks_bertopic_df['speech_id'].nunique()}")

Validasi kolom wajib berhasil.
Jumlah dokumen chunk: 74
Jumlah pidato unik: 6


## 6. Menyiapkan Dokumen untuk BERTopic

Pada tahap ini dilakukan filtering dokumen:

1. Dokumen kosong tidak digunakan.
2. Jika `manual_coding_final.csv` tersedia, chunk dengan `coding_status = EXCLUDED` dikeluarkan dari proses modeling.
3. Semua dokumen yang digunakan tetap disimpan dalam tabel modeling agar dapat diaudit kembali.

Konfigurasi default:

```python
USE_MANUAL_EXCLUSION_FILTER = True
```

In [6]:
# ============================================================
# Prepare Modeling Corpus
# ============================================================

USE_MANUAL_EXCLUSION_FILTER = True

modeling_df = chunks_bertopic_df.copy()

modeling_df["text"] = modeling_df["text"].astype(str).str.strip()
modeling_df["include_for_bertopic"] = True
modeling_df["exclusion_reason"] = ""

# Hapus dokumen kosong jika ada.
empty_text_mask = modeling_df["text"].str.len() == 0
modeling_df.loc[empty_text_mask, "include_for_bertopic"] = False
modeling_df.loc[empty_text_mask, "exclusion_reason"] = "EMPTY_TEXT"

# Join dengan manual coding final jika tersedia.
if manual_coding_final_df is not None:
    manual_reference_columns = [
        "chunk_id",
        "manual_open_code_1",
        "manual_open_code_2",
        "manual_open_code_3",
        "manual_axial_category",
        "manual_selective_theme",
        "coding_status",
        "coder_notes"
    ]

    available_manual_reference_columns = [
        col for col in manual_reference_columns
        if col in manual_coding_final_df.columns
    ]

    manual_reference_df = manual_coding_final_df[available_manual_reference_columns].copy()
    manual_reference_df = manual_reference_df.rename(columns={"chunk_id": "doc_id"})

    modeling_df = modeling_df.merge(
        manual_reference_df,
        on="doc_id",
        how="left",
        validate="one_to_one"
    )

    if USE_MANUAL_EXCLUSION_FILTER and "coding_status" in modeling_df.columns:
        excluded_mask = modeling_df["coding_status"].fillna("").str.upper().eq("EXCLUDED")
        modeling_df.loc[excluded_mask, "include_for_bertopic"] = False
        modeling_df.loc[excluded_mask, "exclusion_reason"] = "MANUAL_CODING_EXCLUDED"

docs_modeling_df = modeling_df[modeling_df["include_for_bertopic"]].copy()
docs_modeling_df = docs_modeling_df.reset_index(drop=True)

if docs_modeling_df.empty:
    raise ValueError("Tidak ada dokumen yang tersisa untuk BERTopic setelah filtering.")

docs = docs_modeling_df["text"].astype(str).tolist()

print("Persiapan corpus selesai.")
print(f"Jumlah dokumen awal       : {len(chunks_bertopic_df)}")
print(f"Jumlah dokumen modeling   : {len(docs_modeling_df)}")
print(f"Jumlah dokumen dikecualikan: {len(chunks_bertopic_df) - len(docs_modeling_df)}")

if len(docs_modeling_df) < 20:
    print("Peringatan: jumlah dokumen modeling kurang dari 20. Hasil BERTopic mungkin kurang stabil.")

display(
    modeling_df[
        [
            "doc_id",
            "speech_id",
            "chunk_order",
            "include_for_bertopic",
            "exclusion_reason",
            "chunk_word_count",
            "text"
        ]
    ].head(10)
)

Persiapan corpus selesai.
Jumlah dokumen awal       : 74
Jumlah dokumen modeling   : 71
Jumlah dokumen dikecualikan: 3


,doc_id,speech_id,chunk_order,include_for_bertopic,exclusion_reason,chunk_word_count,text
0,SPCH_001_BRICS_LEADERS_CHK_001,SPCH_001_BRICS_LEADERS,1,True,,227,Distinguished Leaders of BRICS. It is indeed a...
1,SPCH_001_BRICS_LEADERS_CHK_002,SPCH_001_BRICS_LEADERS,2,True,,42,"We consider now, this is the time that BRICS m..."
2,SPCH_002_PANEN_RAYA_CHK_001,SPCH_002_PANEN_RAYA,1,True,,268,Bismillahirrahmanirrahim. Assalamu'alaikum war...
3,SPCH_002_PANEN_RAYA_CHK_002,SPCH_002_PANEN_RAYA,2,True,,224,"Yang saya hormati, para Dirut BUMN yang berken..."
4,SPCH_002_PANEN_RAYA_CHK_003,SPCH_002_PANEN_RAYA,3,False,MANUAL_CODING_EXCLUDED,242,"Dirut PT Berdikari Saudara Maryadi, Dirut Sang..."
5,SPCH_002_PANEN_RAYA_CHK_004,SPCH_002_PANEN_RAYA,4,True,,222,"Walaupun selalu, kita selalu ingat saudara-sau..."
6,SPCH_002_PANEN_RAYA_CHK_005,SPCH_002_PANEN_RAYA,5,True,,233,"Dari dulu saya mengerti hal ini, tetapi saya t..."
7,SPCH_002_PANEN_RAYA_CHK_006,SPCH_002_PANEN_RAYA,6,True,,234,"Karena itu, saya berjuang terus, saya dituduh ..."
8,SPCH_002_PANEN_RAYA_CHK_007,SPCH_002_PANEN_RAYA,7,True,,233,"Dan, saya tidak habis pikir, puluhan tahun par..."
9,SPCH_002_PANEN_RAYA_CHK_008,SPCH_002_PANEN_RAYA,8,True,,221,"Politik di Indonesia ini pengorbanan, ingin me..."


## 7. Setup dan Validasi Dependency BERTopic

BERTopic membutuhkan beberapa library tambahan:

```text
bertopic
sentence-transformers
umap-learn
hdbscan
scikit-learn
plotly
```

Jika library belum tersedia, cell ini dapat menginstal package secara otomatis.

Catatan:

- Proses instalasi membutuhkan koneksi internet.
- Model embedding `paraphrase-multilingual-MiniLM-L12-v2` juga perlu diunduh saat pertama kali dijalankan.
- Jika koneksi internet bermasalah, instal package secara manual melalui terminal.

In [7]:
# ============================================================
# Dependency Check dan Optional Auto Install
# ============================================================

AUTO_INSTALL_MISSING_PACKAGES = True

REQUIRED_PACKAGES = {
    "bertopic": "bertopic",
    "sentence_transformers": "sentence-transformers",
    "umap": "umap-learn",
    "hdbscan": "hdbscan",
    "sklearn": "scikit-learn",
    "plotly": "plotly"
}

def is_module_available(module_name):
    """
    Mengecek apakah module tersedia.
    """
    return importlib.util.find_spec(module_name) is not None


missing_packages = []

for module_name, package_name in REQUIRED_PACKAGES.items():
    if not is_module_available(module_name):
        missing_packages.append(package_name)

if missing_packages:
    print("Package berikut belum tersedia:")
    for package in missing_packages:
        print(f"- {package}")

    if AUTO_INSTALL_MISSING_PACKAGES:
        print("\nAUTO_INSTALL_MISSING_PACKAGES=True. Instalasi akan dijalankan.")
        install_command = [sys.executable, "-m", "pip", "install", "--upgrade"] + missing_packages
        print("Menjalankan command:")
        print(" ".join(install_command))

        subprocess.check_call(install_command)
        print("Instalasi package selesai. Jika import masih gagal, restart kernel lalu jalankan ulang notebook.")
    else:
        raise ImportError(
            "Terdapat package yang belum tersedia. "
            "Set AUTO_INSTALL_MISSING_PACKAGES=True atau install manual melalui terminal."
        )
else:
    print("Seluruh package utama BERTopic sudah tersedia.")

Seluruh package utama BERTopic sudah tersedia.


## 8. Import Library BERTopic

Cell ini mengimpor library utama untuk modeling.

Jika cell ini gagal karena dependency, restart kernel setelah instalasi package, lalu jalankan ulang notebook.

In [8]:
# ============================================================
# Import Library BERTopic
# ============================================================

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.vectorizers import ClassTfidfTransformer

import umap
import hdbscan
import plotly.express as px

print("Library BERTopic berhasil di-import.")

Library BERTopic berhasil di-import.


## 9. Konfigurasi Model BERTopic

Karena jumlah dokumen relatif kecil, konfigurasi dibuat lebih ringan:

- `min_cluster_size = 3`
- `min_samples = 1`
- `n_neighbors = 5`
- `n_components = 5`

Model embedding yang digunakan:

```text
paraphrase-multilingual-MiniLM-L12-v2
```

Model tersebut dipilih karena dataset berisi bahasa Indonesia dan bahasa Inggris.

In [9]:
# ============================================================
# Konfigurasi Model BERTopic
# ============================================================

RANDOM_STATE = 42

BERTOPIC_CONFIG = {
    "embedding_model_name": "paraphrase-multilingual-MiniLM-L12-v2",
    "umap_n_neighbors": min(5, max(2, len(docs) - 1)),
    "umap_n_components": 5 if len(docs) >= 10 else 2,
    "umap_min_dist": 0.0,
    "umap_metric": "cosine",
    "hdbscan_min_cluster_size": 3 if len(docs) >= 10 else 2,
    "hdbscan_min_samples": 1,
    "hdbscan_metric": "euclidean",
    "vectorizer_ngram_range": (1, 2),
    "vectorizer_min_df": 1,
    "ctfidf_reduce_frequent_words": True,
    "calculate_probabilities": True,
    "random_state": RANDOM_STATE
}

INDONESIAN_STOPWORDS = [
    "yang", "dan", "di", "ke", "dari", "dengan", "untuk", "pada", "dalam",
    "ini", "itu", "adalah", "akan", "atau", "juga", "kita", "kami", "saya",
    "mereka", "sebagai", "karena", "telah", "tidak", "ada", "bisa", "dapat",
    "harus", "lebih", "sangat", "para", "saudara", "sekalian", "terima",
    "kasih", "bapak", "ibu", "hadirin", "hormat", "presiden", "republik",
    "indonesia", "prabowo", "subianto"
]

ENGLISH_STOPWORDS = [
    "the", "and", "of", "to", "in", "for", "with", "on", "as", "is", "are",
    "was", "were", "be", "been", "being", "we", "our", "us", "you", "your",
    "they", "their", "this", "that", "these", "those", "will", "would",
    "can", "could", "should", "must", "not", "have", "has", "had",
    "indonesia", "president", "ladies", "gentlemen", "excellencies",
    "distinguished", "thank"
]

CUSTOM_STOPWORDS = sorted(set(INDONESIAN_STOPWORDS + ENGLISH_STOPWORDS))

print("Konfigurasi BERTopic:")
print(json.dumps(BERTOPIC_CONFIG, indent=2, ensure_ascii=False, default=str))
print(f"Jumlah custom stopwords: {len(CUSTOM_STOPWORDS)}")

Konfigurasi BERTopic:
{
  "embedding_model_name": "paraphrase-multilingual-MiniLM-L12-v2",
  "umap_n_neighbors": 5,
  "umap_n_components": 5,
  "umap_min_dist": 0.0,
  "umap_metric": "cosine",
  "hdbscan_min_cluster_size": 3,
  "hdbscan_min_samples": 1,
  "hdbscan_metric": "euclidean",
  "vectorizer_ngram_range": [
    1,
    2
  ],
  "vectorizer_min_df": 1,
  "ctfidf_reduce_frequent_words": true,
  "calculate_probabilities": true,
  "random_state": 42
}
Jumlah custom stopwords: 86


## 10. Membuat Embedding Model dan Komponen BERTopic

Cell ini menyiapkan:

1. SentenceTransformer untuk embedding multilingual.
2. UMAP untuk reduksi dimensi.
3. HDBSCAN untuk clustering.
4. CountVectorizer untuk ekstraksi kata.
5. ClassTfidfTransformer untuk representasi topik.

In [10]:
# ============================================================
# Build BERTopic Components
# ============================================================

embedding_model = SentenceTransformer(BERTOPIC_CONFIG["embedding_model_name"])

umap_model = umap.UMAP(
    n_neighbors=BERTOPIC_CONFIG["umap_n_neighbors"],
    n_components=BERTOPIC_CONFIG["umap_n_components"],
    min_dist=BERTOPIC_CONFIG["umap_min_dist"],
    metric=BERTOPIC_CONFIG["umap_metric"],
    random_state=BERTOPIC_CONFIG["random_state"]
)

hdbscan_model = hdbscan.HDBSCAN(
    min_cluster_size=BERTOPIC_CONFIG["hdbscan_min_cluster_size"],
    min_samples=BERTOPIC_CONFIG["hdbscan_min_samples"],
    metric=BERTOPIC_CONFIG["hdbscan_metric"],
    prediction_data=True
)

vectorizer_model = CountVectorizer(
    stop_words=CUSTOM_STOPWORDS,
    ngram_range=BERTOPIC_CONFIG["vectorizer_ngram_range"],
    min_df=BERTOPIC_CONFIG["vectorizer_min_df"]
)

ctfidf_model = ClassTfidfTransformer(
    reduce_frequent_words=BERTOPIC_CONFIG["ctfidf_reduce_frequent_words"]
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    calculate_probabilities=BERTOPIC_CONFIG["calculate_probabilities"],
    verbose=True
)

print("Komponen BERTopic berhasil dibuat.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Komponen BERTopic berhasil dibuat.


## 11. Training BERTopic Model

Cell ini menjalankan proses utama:

```python
topics, probabilities = topic_model.fit_transform(docs)
```

Hasilnya adalah topic assignment untuk setiap chunk.

In [11]:
# ============================================================
# Fit BERTopic Model
# ============================================================

topics, probabilities = topic_model.fit_transform(docs)

print("Training BERTopic selesai.")
print(f"Jumlah dokumen modeling: {len(docs)}")
print(f"Jumlah topic assignment: {len(topics)}")

topic_series = pd.Series(topics, name="bertopic_topic_id")
display(
    topic_series
    .value_counts(dropna=False)
    .sort_index()
    .reset_index()
    .rename(columns={"index": "bertopic_topic_id", "bertopic_topic_id": "document_count"})
)

2026-06-11 23:36:38,736 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

2026-06-11 23:36:44,983 - BERTopic - Embedding - Completed ✓
2026-06-11 23:36:44,983 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-11 23:37:07,419 - BERTopic - Dimensionality - Completed ✓
2026-06-11 23:37:07,419 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-11 23:37:07,452 - BERTopic - Cluster - Completed ✓
2026-06-11 23:37:07,452 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-11 23:37:07,554 - BERTopic - Representation - Completed ✓


Training BERTopic selesai.
Jumlah dokumen modeling: 71
Jumlah topic assignment: 71


,document_count,count
0,-1,2
1,0,13
2,1,7
3,2,6
4,3,6
5,4,5
6,5,5
7,6,4
8,7,4
9,8,4


## 12. Membuat Document Topic Table

Tabel ini menyimpan hasil assignment topik untuk setiap dokumen chunk.

Kolom utama:

- `doc_id`
- `speech_id`
- `text`
- `bertopic_topic_id`
- `bertopic_topic_probability`

In [12]:
# ============================================================
# Create Document Topics Output
# ============================================================

def extract_probability(probabilities, row_index):
    """
    Mengekstrak probabilitas tertinggi untuk dokumen.
    """
    if probabilities is None:
        return np.nan

    arr = np.asarray(probabilities)

    if arr.ndim == 1:
        try:
            return float(arr[row_index])
        except Exception:
            return np.nan

    if arr.ndim == 2:
        if arr.shape[1] == 0:
            return np.nan
        return float(np.max(arr[row_index]))

    return np.nan


bertopic_document_topics_df = docs_modeling_df.copy()
bertopic_document_topics_df["bertopic_topic_id"] = topics
bertopic_document_topics_df["bertopic_topic_probability"] = [
    extract_probability(probabilities, idx)
    for idx in range(len(bertopic_document_topics_df))
]

# Tambahkan nama topik dari BERTopic.
topic_info_temp_df = topic_model.get_topic_info()
topic_name_map = dict(zip(topic_info_temp_df["Topic"], topic_info_temp_df["Name"]))

bertopic_document_topics_df["bertopic_topic_name"] = (
    bertopic_document_topics_df["bertopic_topic_id"].map(topic_name_map)
)

# Buat status outlier.
bertopic_document_topics_df["is_outlier_topic"] = (
    bertopic_document_topics_df["bertopic_topic_id"] == -1
)

display(bertopic_document_topics_df.head())

print(f"Jumlah dokumen bertopik: {len(bertopic_document_topics_df)}")
print(f"Jumlah outlier topic (-1): {int(bertopic_document_topics_df['is_outlier_topic'].sum())}")

,doc_id,speech_id,file_name,speech_title_from_filename,forum_scope_inferred,event_date,language_estimate,chunk_order,text,chunk_word_count,...,manual_open_code_2,manual_open_code_3,manual_axial_category,manual_selective_theme,coding_status,coder_notes,bertopic_topic_id,bertopic_topic_probability,bertopic_topic_name,is_outlier_topic
0,SPCH_001_BRICS_LEADERS_CHK_001,SPCH_001_BRICS_LEADERS,NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt,Brics Leaders,international,2025-09-08,en,1,Distinguished Leaders of BRICS. It is indeed a...,227,...,Anti-korupsi dan penegakan hukum,NaN,Diplomasi dan tatanan global,"Diplomasi, perdamaian, dan keadilan global",REVIEWED,NaN,9,1.000000,9_brics_best_danantara_now,False
1,SPCH_001_BRICS_LEADERS_CHK_002,SPCH_001_BRICS_LEADERS,NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt,Brics Leaders,international,2025-09-08,en,2,"We consider now, this is the time that BRICS m...",42,...,NaN,NaN,Diplomasi dan tatanan global,"Diplomasi, perdamaian, dan keadilan global",REVIEWED,NaN,9,1.000000,9_brics_best_danantara_now,False
2,SPCH_002_PANEN_RAYA_CHK_001,SPCH_002_PANEN_RAYA,NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt,Panen Raya,national,2026-01-07,id,1,Bismillahirrahmanirrahim. Assalamu'alaikum war...,268,...,Pemerataan kesejahteraan dan pengentasan kemis...,NaN,Ketahanan nasional dan kedaulatan strategis,Kedaulatan dan kemandirian nasional,REVIEWED,NaN,0,0.598132,0_bupati_hadir_menteri_hormati,False
3,SPCH_002_PANEN_RAYA_CHK_002,SPCH_002_PANEN_RAYA,NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt,Panen Raya,national,2026-01-07,id,2,"Yang saya hormati, para Dirut BUMN yang berken...",224,...,Pemerataan kesejahteraan dan pengentasan kemis...,NaN,Ketahanan nasional dan kedaulatan strategis,Kedaulatan dan kemandirian nasional,REVIEWED,NaN,0,0.476953,0_bupati_hadir_menteri_hormati,False
4,SPCH_002_PANEN_RAYA_CHK_004,SPCH_002_PANEN_RAYA,NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt,Panen Raya,national,2026-01-07,id,4,"Walaupun selalu, kita selalu ingat saudara-sau...",222,...,Kedaulatan pangan dan swasembada,"Investasi, industrialisasi, dan pertumbuhan ek...",Identitas nasional dan kepemimpinan,Kedaulatan dan kemandirian nasional,REVIEWED,NaN,10,1.000000,10_sudah_proyek_mengerti_juta,False


Jumlah dokumen bertopik: 71
Jumlah outlier topic (-1): 2


## 13. Membuat Topic Info dan Topic Keywords

Tabel `bertopic_topic_info.csv` berisi ringkasan topik.  
Tabel `bertopic_topic_keywords.csv` berisi kata kunci setiap topik beserta skor c-TF-IDF.

In [13]:
# ============================================================
# Topic Info dan Topic Keywords
# ============================================================

bertopic_topic_info_df = topic_model.get_topic_info().copy()

def build_auto_topic_label(topic_id, top_n=5):
    """
    Membuat label topik otomatis dari kata kunci teratas.
    """
    if topic_id == -1:
        return "Outlier / tidak terklaster"

    topic_words = topic_model.get_topic(topic_id)

    if not topic_words:
        return "Topik tanpa keyword"

    keywords = [word for word, score in topic_words[:top_n]]

    return " / ".join(keywords)


bertopic_topic_info_df["auto_topic_label"] = bertopic_topic_info_df["Topic"].apply(build_auto_topic_label)
bertopic_topic_info_df["interpreted_topic_label"] = ""
bertopic_topic_info_df["interpretation_notes"] = ""

topic_keyword_records = []

for topic_id in bertopic_topic_info_df["Topic"].tolist():
    topic_words = topic_model.get_topic(topic_id)

    if topic_words is None:
        continue

    for rank, (word, score) in enumerate(topic_words, start=1):
        topic_keyword_records.append({
            "bertopic_topic_id": topic_id,
            "keyword_rank": rank,
            "keyword": word,
            "keyword_score": float(score),
            "auto_topic_label": build_auto_topic_label(topic_id)
        })

bertopic_topic_keywords_df = pd.DataFrame(topic_keyword_records)

display(bertopic_topic_info_df)
display(bertopic_topic_keywords_df.head(30))

,Topic,Count,Name,Representation,Representative_Docs,auto_topic_label,interpreted_topic_label,interpretation_notes
0,-1,2,-1_melindungi_ancaman_melindungi ancaman_pertu...,"[melindungi, ancaman, melindungi ancaman, pert...","[Nah, ini teori, tapi kenyataannya menetesnya ...",Outlier / tidak terklaster,,
1,0,13,0_bupati_hadir_menteri_hormati,"[bupati, hadir, menteri, hormati, wali kota, w...",[Bismillahirrahmanirrahim. Assalamu'alaikum wa...,bupati / hadir / menteri / hormati / wali kota,,
2,1,7,1_kampus_stability_growth_peace stability,"[kampus, stability, growth, peace stability, o...","[Human folly, fueled by fear, racism, hatred, ...",kampus / stability / growth / peace stability ...,,
3,2,6,2_energi_menghasilkan_kesulitan_hasilkan,"[energi, menghasilkan, kesulitan, hasilkan, te...",[Juga kemampuan kita di energi dari air dan ju...,energi / menghasilkan / kesulitan / hasilkan /...,,
4,3,6,3_000_rice_years_000 murid,"[000, rice, years, 000 murid, 83, 83 000, sasa...",[We choose to answer these challenges directly...,000 / rice / years / 000 murid / 83,,
5,4,5,4_kemerdekaan_dia_perang_mau,"[kemerdekaan, dia, perang, mau, enggak, rakyat...","[Saya ndak tanya, saya enggak mau tahu, kita s...",kemerdekaan / dia / perang / mau / enggak,,
6,5,5,5_swasembada_amran_harga_tokoh,"[swasembada, amran, harga, tokoh, kulitnya, ja...","[Dirut Agrinas Pangan, ini orang dari Timor Ti...",swasembada / amran / harga / tokoh / kulitnya,,
7,6,4,6_anak_orang tuamu_tuamu_son,"[anak, orang tuamu, tuamu, son, kau, ya, anak ...","[Jadi, apa adanya. Kalau tidak benar, saya bil...",anak / orang tuamu / tuamu / son / kau,,
8,7,4,7_united nations_nations_united_peace,"[united nations, nations, united, peace, all, ...",[We must maintain and be a part of a future fu...,united nations / nations / united / peace / all,,
9,8,4,8_koperasi_meals_day_guru,"[koperasi, meals, day, guru, desember, ribu, k...",[We saved 18 billion dollars by stopping ineff...,koperasi / meals / day / guru / desember,,


,bertopic_topic_id,keyword_rank,keyword,keyword_score,auto_topic_label
0,-1,1,melindungi,0.583823,Outlier / tidak terklaster
1,-1,2,ancaman,0.557235,Outlier / tidak terklaster
2,-1,3,melindungi ancaman,0.525533,Outlier / tidak terklaster
3,-1,4,pertumbuhan,0.454981,Outlier / tidak terklaster
4,-1,5,merdeka,0.446433,Outlier / tidak terklaster
5,-1,6,who save,0.443683,Outlier / tidak terklaster
6,-1,7,berani melihat,0.407076,Outlier / tidak terklaster
7,-1,8,tujuan,0.407076,Outlier / tidak terklaster
8,-1,9,from,0.402584,Outlier / tidak terklaster
9,-1,10,kesejahteraan,0.386014,Outlier / tidak terklaster


## 14. Distribusi Topik

Cell ini membuat distribusi jumlah dokumen per topik, termasuk jumlah dokumen per forum nasional dan internasional jika kolom tersedia.

In [14]:
# ============================================================
# Topic Distribution
# ============================================================

topic_distribution_df = (
    bertopic_document_topics_df
    .groupby(["bertopic_topic_id", "bertopic_topic_name", "is_outlier_topic"], dropna=False)
    .agg(
        document_count=("doc_id", "count"),
        avg_topic_probability=("bertopic_topic_probability", "mean"),
        avg_chunk_word_count=("chunk_word_count", "mean")
    )
    .reset_index()
)

topic_distribution_df["avg_topic_probability"] = topic_distribution_df["avg_topic_probability"].round(4)
topic_distribution_df["avg_chunk_word_count"] = topic_distribution_df["avg_chunk_word_count"].round(2)

topic_distribution_df = topic_distribution_df.sort_values(
    by=["is_outlier_topic", "document_count"],
    ascending=[True, False]
).reset_index(drop=True)

display(topic_distribution_df)

if "forum_scope_inferred" in bertopic_document_topics_df.columns:
    print("Distribusi topik berdasarkan forum_scope_inferred:")
    topic_by_forum_df = (
        bertopic_document_topics_df
        .groupby(["bertopic_topic_id", "forum_scope_inferred"], dropna=False)
        .agg(document_count=("doc_id", "count"))
        .reset_index()
        .sort_values(["bertopic_topic_id", "document_count"], ascending=[True, False])
    )
    display(topic_by_forum_df)
else:
    topic_by_forum_df = pd.DataFrame()

,bertopic_topic_id,bertopic_topic_name,is_outlier_topic,document_count,avg_topic_probability,avg_chunk_word_count
0,0,0_bupati_hadir_menteri_hormati,False,13,0.7301,229.85
1,1,1_kampus_stability_growth_peace stability,False,7,0.5441,228.57
2,2,2_energi_menghasilkan_kesulitan_hasilkan,False,6,0.5945,229.33
3,3,3_000_rice_years_000 murid,False,6,0.6481,230.50
4,4,4_kemerdekaan_dia_perang_mau,False,5,0.6708,233.20
5,5,5_swasembada_amran_harga_tokoh,False,5,0.6554,232.00
6,6,6_anak_orang tuamu_tuamu_son,False,4,0.7934,219.50
7,7,7_united nations_nations_united_peace,False,4,0.8138,203.50
8,8,8_koperasi_meals_day_guru,False,4,0.8526,242.75
9,9,9_brics_best_danantara_now,False,3,1.0000,167.00


Distribusi topik berdasarkan forum_scope_inferred:


,bertopic_topic_id,forum_scope_inferred,document_count
0,-1,international,1
1,-1,national,1
3,0,national,11
2,0,international,2
4,1,international,6
5,1,national,1
7,2,national,5
6,2,international,1
8,3,international,4
9,3,national,2


## 15. Menggabungkan Hasil BERTopic dengan Referensi Manual Coding

Output ini belum menjadi analisis komparatif final.  
Tujuannya hanya menyiapkan file gabungan agar Tahap 06 lebih mudah dilakukan.

In [15]:
# ============================================================
# Merge BERTopic Result with Manual Reference
# ============================================================

bertopic_document_topics_with_manual_reference_df = bertopic_document_topics_df.copy()

manual_reference_available = all(
    col in bertopic_document_topics_with_manual_reference_df.columns
    for col in ["manual_open_code_1", "manual_axial_category", "manual_selective_theme", "coding_status"]
)

if manual_reference_available:
    print("Kolom referensi manual coding tersedia pada output document topics.")
else:
    print("Kolom referensi manual coding tidak lengkap. Tahap 06 masih dapat dilakukan jika manual_coding_final.csv tersedia.")

preview_cols = [
    "doc_id",
    "speech_id",
    "chunk_order",
    "bertopic_topic_id",
    "bertopic_topic_name",
    "bertopic_topic_probability",
    "manual_open_code_1",
    "manual_axial_category",
    "manual_selective_theme",
    "coding_status",
    "text"
]

available_preview_cols = [
    col for col in preview_cols
    if col in bertopic_document_topics_with_manual_reference_df.columns
]

display(bertopic_document_topics_with_manual_reference_df[available_preview_cols].head(10))

Kolom referensi manual coding tersedia pada output document topics.


,doc_id,speech_id,chunk_order,bertopic_topic_id,bertopic_topic_name,bertopic_topic_probability,manual_open_code_1,manual_axial_category,manual_selective_theme,coding_status,text
0,SPCH_001_BRICS_LEADERS_CHK_001,SPCH_001_BRICS_LEADERS,1,9,9_brics_best_danantara_now,1.000000,Diplomasi multilateral dan kerja sama internas...,Diplomasi dan tatanan global,"Diplomasi, perdamaian, dan keadilan global",REVIEWED,Distinguished Leaders of BRICS. It is indeed a...
1,SPCH_001_BRICS_LEADERS_CHK_002,SPCH_001_BRICS_LEADERS,2,9,9_brics_best_danantara_now,1.000000,Diplomasi multilateral dan kerja sama internas...,Diplomasi dan tatanan global,"Diplomasi, perdamaian, dan keadilan global",REVIEWED,"We consider now, this is the time that BRICS m..."
2,SPCH_002_PANEN_RAYA_CHK_001,SPCH_002_PANEN_RAYA,1,0,0_bupati_hadir_menteri_hormati,0.598132,Kedaulatan pangan dan swasembada,Ketahanan nasional dan kedaulatan strategis,Kedaulatan dan kemandirian nasional,REVIEWED,Bismillahirrahmanirrahim. Assalamu'alaikum war...
3,SPCH_002_PANEN_RAYA_CHK_002,SPCH_002_PANEN_RAYA,2,0,0_bupati_hadir_menteri_hormati,0.476953,Kedaulatan pangan dan swasembada,Ketahanan nasional dan kedaulatan strategis,Kedaulatan dan kemandirian nasional,REVIEWED,"Yang saya hormati, para Dirut BUMN yang berken..."
4,SPCH_002_PANEN_RAYA_CHK_004,SPCH_002_PANEN_RAYA,4,10,10_sudah_proyek_mengerti_juta,1.000000,Persatuan nasional dan kepercayaan diri bangsa,Identitas nasional dan kepemimpinan,Kedaulatan dan kemandirian nasional,REVIEWED,"Walaupun selalu, kita selalu ingat saudara-sau..."
5,SPCH_002_PANEN_RAYA_CHK_005,SPCH_002_PANEN_RAYA,5,11,11_akal_berjuang_masuk akal_leader,1.000000,Pemerataan kesejahteraan dan pengentasan kemis...,Kesejahteraan sosial dan pemerataan pembangunan,Pembangunan manusia dan keadilan sosial,REVIEWED,"Dari dulu saya mengerti hal ini, tetapi saya t..."
6,SPCH_002_PANEN_RAYA_CHK_006,SPCH_002_PANEN_RAYA,6,4,4_kemerdekaan_dia_perang_mau,1.000000,"Koperasi, desa, dan ekonomi akar rumput",Transformasi ekonomi dan pembangunan produktif,Transformasi ekonomi dan pembangunan nasional,REVIEWED,"Karena itu, saya berjuang terus, saya dituduh ..."
7,SPCH_002_PANEN_RAYA_CHK_007,SPCH_002_PANEN_RAYA,7,5,5_swasembada_amran_harga_tokoh,0.059700,Pemerataan kesejahteraan dan pengentasan kemis...,Kesejahteraan sosial dan pemerataan pembangunan,Pembangunan manusia dan keadilan sosial,REVIEWED,"Dan, saya tidak habis pikir, puluhan tahun par..."
8,SPCH_002_PANEN_RAYA_CHK_008,SPCH_002_PANEN_RAYA,8,4,4_kemerdekaan_dia_perang_mau,0.148686,Pemerataan kesejahteraan dan pengentasan kemis...,Kesejahteraan sosial dan pemerataan pembangunan,Pembangunan manusia dan keadilan sosial,REVIEWED,"Politik di Indonesia ini pengorbanan, ingin me..."
9,SPCH_002_PANEN_RAYA_CHK_009,SPCH_002_PANEN_RAYA,9,2,2_energi_menghasilkan_kesulitan_hasilkan,0.200851,Kedaulatan pangan dan swasembada,Ketahanan nasional dan kedaulatan strategis,Kedaulatan dan kemandirian nasional,REVIEWED,"Jadi, Saudara-saudara, bersyukurlah kalau kau ..."


## 16. Ringkasan Modeling BERTopic

Ringkasan ini digunakan sebagai audit hasil modeling.

In [16]:
# ============================================================
# BERTopic Modeling Summary
# ============================================================

non_outlier_topic_ids = [
    topic_id for topic_id in sorted(set(topics))
    if topic_id != -1
]

outlier_count = int((bertopic_document_topics_df["bertopic_topic_id"] == -1).sum())
outlier_ratio = outlier_count / len(bertopic_document_topics_df) if len(bertopic_document_topics_df) > 0 else 0

summary_records = [
    {
        "metric": "input_document_count_before_filter",
        "value": int(len(chunks_bertopic_df)),
        "description": "Jumlah dokumen chunk sebelum filtering."
    },
    {
        "metric": "modeling_document_count",
        "value": int(len(bertopic_document_topics_df)),
        "description": "Jumlah dokumen chunk yang digunakan untuk BERTopic."
    },
    {
        "metric": "excluded_document_count",
        "value": int(len(chunks_bertopic_df) - len(bertopic_document_topics_df)),
        "description": "Jumlah dokumen chunk yang tidak digunakan untuk BERTopic."
    },
    {
        "metric": "topic_count_excluding_outlier",
        "value": int(len(non_outlier_topic_ids)),
        "description": "Jumlah topik valid di luar topik outlier -1."
    },
    {
        "metric": "outlier_document_count",
        "value": int(outlier_count),
        "description": "Jumlah dokumen dengan topic_id -1."
    },
    {
        "metric": "outlier_ratio",
        "value": round(float(outlier_ratio), 4),
        "description": "Rasio dokumen outlier terhadap total dokumen modeling."
    },
    {
        "metric": "average_topic_probability",
        "value": round(float(bertopic_document_topics_df["bertopic_topic_probability"].mean()), 4),
        "description": "Rata-rata probabilitas topik tertinggi per dokumen."
    },
    {
        "metric": "embedding_model_name",
        "value": BERTOPIC_CONFIG["embedding_model_name"],
        "description": "Model sentence embedding yang digunakan."
    },
    {
        "metric": "manual_exclusion_filter_used",
        "value": bool(USE_MANUAL_EXCLUSION_FILTER),
        "description": "Menunjukkan apakah filtering EXCLUDED dari manual coding digunakan."
    }
]

bertopic_modeling_summary_df = pd.DataFrame(summary_records)

display(bertopic_modeling_summary_df)

,metric,value,description
0,input_document_count_before_filter,74,Jumlah dokumen chunk sebelum filtering.
1,modeling_document_count,71,Jumlah dokumen chunk yang digunakan untuk BERT...
2,excluded_document_count,3,Jumlah dokumen chunk yang tidak digunakan untu...
3,topic_count_excluding_outlier,14,Jumlah topik valid di luar topik outlier -1.
4,outlier_document_count,2,Jumlah dokumen dengan topic_id -1.
5,outlier_ratio,0.0282,Rasio dokumen outlier terhadap total dokumen m...
6,average_topic_probability,0.7382,Rata-rata probabilitas topik tertinggi per dok...
7,embedding_model_name,paraphrase-multilingual-MiniLM-L12-v2,Model sentence embedding yang digunakan.
8,manual_exclusion_filter_used,True,Menunjukkan apakah filtering EXCLUDED dari man...


## 17. Visualisasi BERTopic

Visualisasi disimpan dalam format HTML agar dapat dibuka di browser.

Jika visualisasi tertentu gagal karena jumlah topik terlalu sedikit atau ada batasan library, notebook tidak akan berhenti. Sebagai gantinya, file HTML berisi catatan error akan dibuat.

In [17]:
# ============================================================
# BERTopic Visualizations
# ============================================================

topic_barchart_path = REPORT_FIGURE_DIR / "bertopic_topic_barchart.html"
topics_overview_path = REPORT_FIGURE_DIR / "bertopic_topics_overview.html"
topic_distribution_bar_path = REPORT_FIGURE_DIR / "bertopic_topic_distribution_bar.html"

def write_html_message(path, title, message):
    """
    Menulis file HTML sederhana berisi pesan.
    """
    html = f"""
    <html>
    <head><meta charset="utf-8"><title>{title}</title></head>
    <body>
        <h2>{title}</h2>
        <p>{message}</p>
    </body>
    </html>
    """
    path.write_text(html, encoding="utf-8")


# Visualisasi barchart kata kunci per topik.
try:
    top_n_topics = min(10, max(1, len(non_outlier_topic_ids)))
    fig_barchart = topic_model.visualize_barchart(
        top_n_topics=top_n_topics,
        n_words=10
    )
    fig_barchart.write_html(str(topic_barchart_path))
    print(f"Visualisasi barchart berhasil disimpan: {topic_barchart_path}")
except Exception as error:
    write_html_message(
        topic_barchart_path,
        "BERTopic Topic Barchart Tidak Tersedia",
        f"Visualisasi gagal dibuat. Error: {error}"
    )
    print(f"Visualisasi barchart gagal dibuat, file pesan disimpan: {topic_barchart_path}")

# Visualisasi overview antar topik.
try:
    if len(non_outlier_topic_ids) >= 2:
        fig_topics = topic_model.visualize_topics()
        fig_topics.write_html(str(topics_overview_path))
        print(f"Visualisasi topics overview berhasil disimpan: {topics_overview_path}")
    else:
        write_html_message(
            topics_overview_path,
            "BERTopic Topics Overview Tidak Tersedia",
            "Visualisasi overview membutuhkan minimal dua topik non-outlier."
        )
        print(f"Topics overview dilewati karena topik non-outlier kurang dari 2: {topics_overview_path}")
except Exception as error:
    write_html_message(
        topics_overview_path,
        "BERTopic Topics Overview Tidak Tersedia",
        f"Visualisasi gagal dibuat. Error: {error}"
    )
    print(f"Visualisasi topics overview gagal dibuat, file pesan disimpan: {topics_overview_path}")

# Visualisasi distribusi dokumen per topik menggunakan Plotly Express.
try:
    fig_dist = px.bar(
        topic_distribution_df,
        x="bertopic_topic_id",
        y="document_count",
        title="Distribusi Dokumen per Topik BERTopic",
        labels={
            "bertopic_topic_id": "Topic ID",
            "document_count": "Jumlah Dokumen"
        }
    )
    fig_dist.write_html(str(topic_distribution_bar_path))
    print(f"Visualisasi distribusi topik berhasil disimpan: {topic_distribution_bar_path}")
except Exception as error:
    write_html_message(
        topic_distribution_bar_path,
        "BERTopic Topic Distribution Tidak Tersedia",
        f"Visualisasi gagal dibuat. Error: {error}"
    )
    print(f"Visualisasi distribusi topik gagal dibuat, file pesan disimpan: {topic_distribution_bar_path}")

Visualisasi barchart berhasil disimpan: D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\reports\figures\bertopic_topic_barchart.html
Visualisasi topics overview berhasil disimpan: D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\reports\figures\bertopic_topics_overview.html
Visualisasi distribusi topik berhasil disimpan: D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\reports\figures\bertopic_topic_distribution_bar.html


## 18. Optional — Simpan Model BERTopic

Secara default, model tidak disimpan untuk menghindari ukuran file repository yang terlalu besar.

Jika ingin menyimpan model, ubah:

```python
SAVE_BERTOPIC_MODEL = True
```

Catatan:

- Folder model dapat berukuran besar.
- Untuk GitHub public, sebaiknya model besar tidak di-commit.
- Output CSV dan HTML biasanya sudah cukup untuk tugas akademik.

In [18]:
# ============================================================
# Optional Save BERTopic Model
# ============================================================

SAVE_BERTOPIC_MODEL = False

bertopic_model_path = MODELS_DIR / "bertopic_model"

if SAVE_BERTOPIC_MODEL:
    try:
        topic_model.save(
            str(bertopic_model_path),
            serialization="safetensors",
            save_ctfidf=True,
            save_embedding_model=False
        )
        print(f"Model BERTopic berhasil disimpan ke: {bertopic_model_path}")
    except Exception as error:
        print(f"Model BERTopic gagal disimpan. Error: {error}")
else:
    print("Penyimpanan model BERTopic dilewati. SAVE_BERTOPIC_MODEL=False.")

Penyimpanan model BERTopic dilewati. SAVE_BERTOPIC_MODEL=False.


## 19. Validasi Akhir Sebelum Save Output

Cell ini memastikan seluruh output utama sudah terbentuk sebelum disimpan.

In [19]:
# ============================================================
# Final Validation
# ============================================================

if bertopic_document_topics_df.empty:
    raise ValueError("bertopic_document_topics_df kosong.")

if bertopic_topic_info_df.empty:
    raise ValueError("bertopic_topic_info_df kosong.")

if bertopic_topic_keywords_df.empty:
    print("Peringatan: bertopic_topic_keywords_df kosong. Hal ini mungkin terjadi jika model tidak membentuk topik valid.")

if bertopic_modeling_summary_df.empty:
    raise ValueError("bertopic_modeling_summary_df kosong.")

require_columns(
    bertopic_document_topics_df,
    ["doc_id", "speech_id", "text", "bertopic_topic_id", "bertopic_topic_probability"],
    "bertopic_document_topics_df"
)

require_columns(
    bertopic_topic_info_df,
    ["Topic", "Count", "Name"],
    "bertopic_topic_info_df"
)

print("Validasi akhir berhasil. Output siap disimpan.")

Validasi akhir berhasil. Output siap disimpan.


## 20. Menyimpan Output Tahap 05

Output disimpan ke:

```text
data/processed/
reports/tables/
reports/figures/
```

In [20]:
# ============================================================
# Save Output Tahap 05
# ============================================================

bertopic_document_topics_path = PROCESSED_DIR / "bertopic_document_topics.csv"
bertopic_topic_info_path = PROCESSED_DIR / "bertopic_topic_info.csv"
bertopic_topic_keywords_path = PROCESSED_DIR / "bertopic_topic_keywords.csv"
bertopic_document_topics_with_manual_reference_path = PROCESSED_DIR / "bertopic_document_topics_with_manual_reference.csv"

bertopic_modeling_summary_path = REPORT_TABLE_DIR / "bertopic_modeling_summary.csv"
bertopic_topic_distribution_path = REPORT_TABLE_DIR / "bertopic_topic_distribution.csv"
stage05_output_manifest_path = REPORT_TABLE_DIR / "stage05_output_manifest.json"

bertopic_document_topics_df.to_csv(
    bertopic_document_topics_path,
    index=False,
    encoding="utf-8-sig"
)

bertopic_topic_info_df.to_csv(
    bertopic_topic_info_path,
    index=False,
    encoding="utf-8-sig"
)

bertopic_topic_keywords_df.to_csv(
    bertopic_topic_keywords_path,
    index=False,
    encoding="utf-8-sig"
)

bertopic_document_topics_with_manual_reference_df.to_csv(
    bertopic_document_topics_with_manual_reference_path,
    index=False,
    encoding="utf-8-sig"
)

bertopic_modeling_summary_df.to_csv(
    bertopic_modeling_summary_path,
    index=False,
    encoding="utf-8-sig"
)

topic_distribution_df.to_csv(
    bertopic_topic_distribution_path,
    index=False,
    encoding="utf-8-sig"
)

stage05_output_manifest = {
    "stage": "05_bertopic_modeling",
    "input_files": {
        "speech_chunks_for_bertopic": str(INPUT_CHUNKS_BERTOPIC),
        "speech_chunks_master": str(INPUT_CHUNKS_MASTER) if INPUT_CHUNKS_MASTER.exists() else None,
        "manual_coding_final": str(INPUT_MANUAL_CODING_FINAL) if INPUT_MANUAL_CODING_FINAL.exists() else None
    },
    "outputs": {
        "bertopic_document_topics": str(bertopic_document_topics_path),
        "bertopic_topic_info": str(bertopic_topic_info_path),
        "bertopic_topic_keywords": str(bertopic_topic_keywords_path),
        "bertopic_document_topics_with_manual_reference": str(bertopic_document_topics_with_manual_reference_path),
        "bertopic_modeling_summary": str(bertopic_modeling_summary_path),
        "bertopic_topic_distribution": str(bertopic_topic_distribution_path),
        "bertopic_topic_barchart": str(topic_barchart_path),
        "bertopic_topics_overview": str(topics_overview_path),
        "bertopic_topic_distribution_bar": str(topic_distribution_bar_path)
    },
    "input_document_count_before_filter": int(len(chunks_bertopic_df)),
    "modeling_document_count": int(len(bertopic_document_topics_df)),
    "topic_count_excluding_outlier": int(len(non_outlier_topic_ids)),
    "outlier_document_count": int(outlier_count),
    "outlier_ratio": round(float(outlier_ratio), 4),
    "manual_exclusion_filter_used": bool(USE_MANUAL_EXCLUSION_FILTER),
    "bertopic_config": BERTOPIC_CONFIG,
    "created_at": datetime.now().isoformat(timespec="seconds")
}

stage05_output_manifest_path.write_text(
    json.dumps(stage05_output_manifest, indent=2, ensure_ascii=False, default=str),
    encoding="utf-8"
)

print("Output Tahap 05 berhasil disimpan:")
print(f"1. {bertopic_document_topics_path}")
print(f"2. {bertopic_topic_info_path}")
print(f"3. {bertopic_topic_keywords_path}")
print(f"4. {bertopic_document_topics_with_manual_reference_path}")
print(f"5. {bertopic_modeling_summary_path}")
print(f"6. {bertopic_topic_distribution_path}")
print(f"7. {stage05_output_manifest_path}")
print(f"8. {topic_barchart_path}")
print(f"9. {topics_overview_path}")
print(f"10. {topic_distribution_bar_path}")

Output Tahap 05 berhasil disimpan:
1. D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\processed\bertopic_document_topics.csv
2. D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\processed\bertopic_topic_info.csv
3. D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\processed\bertopic_topic_keywords.csv
4. D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\processed\bertopic_document_topics_with_manual_reference.csv
5. D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\reports\tables\bertopic_modeling_summary.csv
6. D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\reports\tables\bertopic_topic_distribution.csv
7. D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\reports\tables\stage05_output_manifest.json
8. D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-spee

## 21. Preview Output Tahap 05

Preview ini digunakan untuk memastikan hasil modeling dapat dibaca sebelum masuk ke Tahap 06.

In [21]:
# ============================================================
# Preview Output Tahap 05
# ============================================================

print("Topic Info:")
display(bertopic_topic_info_df)

print("Topic Distribution:")
display(topic_distribution_df)

print("Document Topics Preview:")
preview_cols = [
    "doc_id",
    "speech_id",
    "forum_scope_inferred",
    "chunk_order",
    "bertopic_topic_id",
    "bertopic_topic_name",
    "bertopic_topic_probability",
    "manual_selective_theme",
    "coding_status",
    "text"
]

available_preview_cols = [
    col for col in preview_cols
    if col in bertopic_document_topics_with_manual_reference_df.columns
]

display(bertopic_document_topics_with_manual_reference_df[available_preview_cols].head(15))

print("Modeling Summary:")
display(bertopic_modeling_summary_df)

Topic Info:


,Topic,Count,Name,Representation,Representative_Docs,auto_topic_label,interpreted_topic_label,interpretation_notes
0,-1,2,-1_melindungi_ancaman_melindungi ancaman_pertu...,"[melindungi, ancaman, melindungi ancaman, pert...","[Nah, ini teori, tapi kenyataannya menetesnya ...",Outlier / tidak terklaster,,
1,0,13,0_bupati_hadir_menteri_hormati,"[bupati, hadir, menteri, hormati, wali kota, w...",[Bismillahirrahmanirrahim. Assalamu'alaikum wa...,bupati / hadir / menteri / hormati / wali kota,,
2,1,7,1_kampus_stability_growth_peace stability,"[kampus, stability, growth, peace stability, o...","[Human folly, fueled by fear, racism, hatred, ...",kampus / stability / growth / peace stability ...,,
3,2,6,2_energi_menghasilkan_kesulitan_hasilkan,"[energi, menghasilkan, kesulitan, hasilkan, te...",[Juga kemampuan kita di energi dari air dan ju...,energi / menghasilkan / kesulitan / hasilkan /...,,
4,3,6,3_000_rice_years_000 murid,"[000, rice, years, 000 murid, 83, 83 000, sasa...",[We choose to answer these challenges directly...,000 / rice / years / 000 murid / 83,,
5,4,5,4_kemerdekaan_dia_perang_mau,"[kemerdekaan, dia, perang, mau, enggak, rakyat...","[Saya ndak tanya, saya enggak mau tahu, kita s...",kemerdekaan / dia / perang / mau / enggak,,
6,5,5,5_swasembada_amran_harga_tokoh,"[swasembada, amran, harga, tokoh, kulitnya, ja...","[Dirut Agrinas Pangan, ini orang dari Timor Ti...",swasembada / amran / harga / tokoh / kulitnya,,
7,6,4,6_anak_orang tuamu_tuamu_son,"[anak, orang tuamu, tuamu, son, kau, ya, anak ...","[Jadi, apa adanya. Kalau tidak benar, saya bil...",anak / orang tuamu / tuamu / son / kau,,
8,7,4,7_united nations_nations_united_peace,"[united nations, nations, united, peace, all, ...",[We must maintain and be a part of a future fu...,united nations / nations / united / peace / all,,
9,8,4,8_koperasi_meals_day_guru,"[koperasi, meals, day, guru, desember, ribu, k...",[We saved 18 billion dollars by stopping ineff...,koperasi / meals / day / guru / desember,,


Topic Distribution:


,bertopic_topic_id,bertopic_topic_name,is_outlier_topic,document_count,avg_topic_probability,avg_chunk_word_count
0,0,0_bupati_hadir_menteri_hormati,False,13,0.7301,229.85
1,1,1_kampus_stability_growth_peace stability,False,7,0.5441,228.57
2,2,2_energi_menghasilkan_kesulitan_hasilkan,False,6,0.5945,229.33
3,3,3_000_rice_years_000 murid,False,6,0.6481,230.50
4,4,4_kemerdekaan_dia_perang_mau,False,5,0.6708,233.20
5,5,5_swasembada_amran_harga_tokoh,False,5,0.6554,232.00
6,6,6_anak_orang tuamu_tuamu_son,False,4,0.7934,219.50
7,7,7_united nations_nations_united_peace,False,4,0.8138,203.50
8,8,8_koperasi_meals_day_guru,False,4,0.8526,242.75
9,9,9_brics_best_danantara_now,False,3,1.0000,167.00


Document Topics Preview:


,doc_id,speech_id,forum_scope_inferred,chunk_order,bertopic_topic_id,bertopic_topic_name,bertopic_topic_probability,manual_selective_theme,coding_status,text
0,SPCH_001_BRICS_LEADERS_CHK_001,SPCH_001_BRICS_LEADERS,international,1,9,9_brics_best_danantara_now,1.000000,"Diplomasi, perdamaian, dan keadilan global",REVIEWED,Distinguished Leaders of BRICS. It is indeed a...
1,SPCH_001_BRICS_LEADERS_CHK_002,SPCH_001_BRICS_LEADERS,international,2,9,9_brics_best_danantara_now,1.000000,"Diplomasi, perdamaian, dan keadilan global",REVIEWED,"We consider now, this is the time that BRICS m..."
2,SPCH_002_PANEN_RAYA_CHK_001,SPCH_002_PANEN_RAYA,national,1,0,0_bupati_hadir_menteri_hormati,0.598132,Kedaulatan dan kemandirian nasional,REVIEWED,Bismillahirrahmanirrahim. Assalamu'alaikum war...
3,SPCH_002_PANEN_RAYA_CHK_002,SPCH_002_PANEN_RAYA,national,2,0,0_bupati_hadir_menteri_hormati,0.476953,Kedaulatan dan kemandirian nasional,REVIEWED,"Yang saya hormati, para Dirut BUMN yang berken..."
4,SPCH_002_PANEN_RAYA_CHK_004,SPCH_002_PANEN_RAYA,national,4,10,10_sudah_proyek_mengerti_juta,1.000000,Kedaulatan dan kemandirian nasional,REVIEWED,"Walaupun selalu, kita selalu ingat saudara-sau..."
5,SPCH_002_PANEN_RAYA_CHK_005,SPCH_002_PANEN_RAYA,national,5,11,11_akal_berjuang_masuk akal_leader,1.000000,Pembangunan manusia dan keadilan sosial,REVIEWED,"Dari dulu saya mengerti hal ini, tetapi saya t..."
6,SPCH_002_PANEN_RAYA_CHK_006,SPCH_002_PANEN_RAYA,national,6,4,4_kemerdekaan_dia_perang_mau,1.000000,Transformasi ekonomi dan pembangunan nasional,REVIEWED,"Karena itu, saya berjuang terus, saya dituduh ..."
7,SPCH_002_PANEN_RAYA_CHK_007,SPCH_002_PANEN_RAYA,national,7,5,5_swasembada_amran_harga_tokoh,0.059700,Pembangunan manusia dan keadilan sosial,REVIEWED,"Dan, saya tidak habis pikir, puluhan tahun par..."
8,SPCH_002_PANEN_RAYA_CHK_008,SPCH_002_PANEN_RAYA,national,8,4,4_kemerdekaan_dia_perang_mau,0.148686,Pembangunan manusia dan keadilan sosial,REVIEWED,"Politik di Indonesia ini pengorbanan, ingin me..."
9,SPCH_002_PANEN_RAYA_CHK_009,SPCH_002_PANEN_RAYA,national,9,2,2_energi_menghasilkan_kesulitan_hasilkan,0.200851,Kedaulatan dan kemandirian nasional,REVIEWED,"Jadi, Saudara-saudara, bersyukurlah kalau kau ..."


Modeling Summary:


,metric,value,description
0,input_document_count_before_filter,74,Jumlah dokumen chunk sebelum filtering.
1,modeling_document_count,71,Jumlah dokumen chunk yang digunakan untuk BERT...
2,excluded_document_count,3,Jumlah dokumen chunk yang tidak digunakan untu...
3,topic_count_excluding_outlier,14,Jumlah topik valid di luar topik outlier -1.
4,outlier_document_count,2,Jumlah dokumen dengan topic_id -1.
5,outlier_ratio,0.0282,Rasio dokumen outlier terhadap total dokumen m...
6,average_topic_probability,0.7382,Rata-rata probabilitas topik tertinggi per dok...
7,embedding_model_name,paraphrase-multilingual-MiniLM-L12-v2,Model sentence embedding yang digunakan.
8,manual_exclusion_filter_used,True,Menunjukkan apakah filtering EXCLUDED dari man...


## 22. Interpretasi Awal

Hal yang perlu diperhatikan dari hasil BERTopic:

1. **Topic ID `-1`** berarti dokumen dianggap outlier atau tidak masuk cluster utama.
2. Jika jumlah outlier terlalu besar, konfigurasi dapat disesuaikan, misalnya memperkecil `min_cluster_size`.
3. `auto_topic_label` hanya label awal dari kata kunci, bukan interpretasi final.
4. Interpretasi final topik perlu dilakukan dengan membaca keyword dan contoh dokumen representatif.
5. File `bertopic_document_topics_with_manual_reference.csv` akan digunakan pada Tahap 06 untuk membandingkan hasil Manual Coding dan BERTopic.

## Output untuk Tahap 06

File yang paling penting untuk Tahap 06:

```text
data/processed/manual_coding_final.csv
data/processed/bertopic_document_topics_with_manual_reference.csv
data/processed/bertopic_topic_info.csv
data/processed/bertopic_topic_keywords.csv
```